This notebook has been used to generate some first tests using GX. It is based on the GX getting started docs. It is not intended to be run externally but only for generating the json configuration which can then be executed from CRON or Airflow.

In [1]:
import great_expectations as gx
context = gx.get_context()
import logging

In [2]:
logging.basicConfig(level=logging.INFO, force = True)

In [3]:
import yaml

In [4]:
with open("datasources/datasources.yml", "r") as ymlfile:
    datasource_config = yaml.full_load(ymlfile)

In [5]:
datasource_config.get("project")

'gfw-google-827'

In [6]:
gx_project = datasource_config.get("project")
#we create a data source for each schema, e.g. pipe_ais_v3_alpha_published
#get datasource if it exists, otherwise create datasource
# WARNING: it's necessary to distinguish because running add_or_update_sql resets the datasource config
# TODO: create feature request to simply get datasource if it already exists
if gx_project in [ds.get("name") for ds in context.list_datasources()]:
    gx_datasource = context.get_datasource(gx_project)
else:
    gx_datasource = context.sources.add_or_update_sql(
        name=gx_project, connection_string=connection_string, create_temp_table=True
    )

In [24]:
for current_asset_name in gx_datasource.get_asset_names():
    current_asset=gx_datasource.get_asset(current_asset_name)
    current_asset_datasource_name=current_asset.batch_metadata.get('datasource_name')
    current_asset_version_number=current_asset.batch_metadata.get('version_number')
    current_asset_version_number_dashed=str(current_asset_version_number).replace(".", "-")

    for current_test_type in ['constraints', 'alerts']:
        current_expectation_suite_name=f"{gx_project}.{current_test_type}.{current_asset_datasource_name}.{current_asset_version_number_dashed}"
        if current_expectation_suite_name not in context.list_expectation_suite_names():
            context.add_or_update_expectation_suite(
                current_expectation_suite_name, 
                meta={
                    'project': gx_project,
                    'test_type': current_test_type,
                    'asset_name': current_asset_name,
                    'datasource_name': current_asset_datasource_name,
                    'version_number': current_asset_version_number
                })
